# Lab 04: Native Pandas Analytics Agent (Gemini + Dynamic Code Execution)
**[🇧🇷 PT-BR]** Agente Analista Nativo (Saneamento Autônomo & EDA Conversacional)  
**[🇺🇸 EN]** Autonomous Data Sanitization & Conversational Pandas Code-Writing Agent  
**Course / Disciplina:** Augmented Analytics & AI-Driven Insights (FIAP MBA)  
**Official Engine / Modelo Oficial:** Gemini 3.8 Flash (`gemini-3.8-flash`) via `google-genai` SDK

---

* 🇧🇷 Este notebook demonstra como criar um agente analista nativo que gera e executa código Python/Pandas em tempo real para higienizar e explorar dados tabulares.
* 🇺🇸 This notebook demonstrates how to build a native analytics agent that synthesizes and executes Python/Pandas code in real time to sanitize and interrogate tabular datasets.

In [ ]:
# 1. Install official Google GenAI SDK and utilities
!pip install -q -U google-genai tabulate

In [ ]:
import pandas as pd
from google import genai
from google.genai import types
import matplotlib.pyplot as plt
import seaborn as sns

# 2. Configure API Key & Output Language
# 🇧🇷 Substitua 'COLE_SUA_API_KEY_AQUI' pela chave do Google AI Studio | 🇺🇸 Replace with your Google AI Studio API Key
API_KEY = 'COLE_SUA_API_KEY_AQUI'
OUTPUT_LANGUAGE = 'Portuguese (Brazil)' # @param ['Portuguese (Brazil)', 'English']
client = genai.Client(api_key=API_KEY)

In [ ]:
# 3. Resilient Self-Healing Dataset Ingestion (Lab 04 - adult.csv)
import os
import urllib.request

DATASET_NAME = 'Lab 04 - adult.csv'
RAW_URL = 'https://raw.githubusercontent.com/rafaelmatsuyama/FIAP-BI-AugmentedAnalytics/main/lab04-pandas-agent/Lab%2004%20-%20adult.csv'

if not os.path.exists(DATASET_NAME):
    print(f"📥 Downloading / Baixando {DATASET_NAME} from official repository...")
    urllib.request.urlretrieve(RAW_URL, DATASET_NAME)

try:
    df = pd.read_csv(DATASET_NAME)
    print("✅ Raw dataset loaded (uncleaned state)! / Dataset carregado no estado bruto (sujo)!")

    contexto_colunas = df.dtypes.to_string()
    amostra_dados = df.head(3).to_markdown()
except Exception as e:
    print(f"❌ Error loading dataset / Erro ao carregar dataset: {e}")

In [ ]:
import re

def sanitizar_dados():
    """
    Uses Gemini to identify and strip leading/trailing whitespace across all categorical columns in global dataframe 'df'.
    """
    prompt = f"""
    You are a Senior Data Engineer. The global dataframe 'df' has the following schema: {contexto_colunas}
    
    Categorical string columns contain leading/trailing whitespace (e.g., ' Female' instead of 'Female'), which breaks exact-match filters.
    
    YOUR TASK:
    1. Write Python code to permanently strip leading/trailing whitespace across all string/object columns of 'df'.
    2. Return ONLY valid executable Python code enclosed in ```python blocks.
    """
    
    try:
        response = client.models.generate_content(
            model='gemini-3.8-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1
            )
        )
        match = re.search(r'```python\n(.*?)\n```', response.text, re.DOTALL)
        if match:
            codigo = match.group(1).strip()
            print("🤖 Data Agent: Running autonomous sanitization / Iniciando saneamento...\n")
            exec(codigo, globals())
            print("✅ Sanitization completed! / Saneamento concluído com sucesso!")
        else:
            print("❌ Failed to extract sanitization code / Erro na extração do código.")
    except Exception as e:
        print(f"⚠️ Sanitization error / Erro no saneamento: {e}")

def chat_com_dados(pergunta):
    """
    Uses Gemini to synthesize Pandas/Matplotlib code from natural language (PT-BR or EN) and execute it locally.
    """
    prompt = f"""
    You are a Senior Python/Pandas Data Analyst.
    The dataframe 'df' is already loaded and sanitized in memory.
    Columns: {contexto_colunas}
    Sample: {amostra_dados}

    IMPORTANT: The 'ABOVE50K' target column is numeric (0 or 1).
    Format any printed text or chart titles in: {OUTPUT_LANGUAGE}.

    USER QUESTION: {pergunta}

    YOUR TASK:
    1. Write ONLY executable Python code to answer the user's question.
    2. Use 'print()' for numerical/tabular outputs and 'plt.show()' for charts.
    3. Return ONLY raw Python code inside ```python blocks.
    """

    try:
        response = client.models.generate_content(
            model='gemini-3.8-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.2
            )
        )
        match = re.search(r'```python\n(.*?)\n```', response.text, re.DOTALL)
        if match:
            codigo = match.group(1).strip()
            print(f"🤖 AI Analyst executing / executando: {pergunta}\n")
            exec(codigo, globals())
        else:
            print("❌ Failed to extract analytical code / Erro na extração do código analítico.")
    except Exception as e:
        print(f"⚠️ Execution error / Erro na execução: {e}")
        if 'codigo' in locals():
            print(f"Attempted code / Código tentado:\n{codigo}")

# International English aliases (both PT-BR and EN function names work seamlessly)
sanitize_data = sanitizar_dados
chat_with_data = chat_com_dados

In [ ]:
# 5. DATA SANITIZATION / SANEAMENTO DOS DADOS (Mandatory before querying / Obrigatório antes do Chat)
sanitizar_dados() # or sanitize_data()

In [ ]:
# 6. ANALYTICAL QUERIES / EXEMPLOS DE USO ANALÍTICO (Supports PT-BR or EN questions):
chat_com_dados("Qual a média de idade das pessoas que ganham acima de 50k?")
# EN Example: chat_with_data("What is the average age of individuals earning above 50k?")

In [ ]:
chat_com_dados("Gere um gráfico de dispersão de Idade vs Horas por Semana colorindo por Sexo.")
# EN Example: chat_with_data("Generate a scatter plot of Age vs Hours per Week colored by Sex.")